In [1]:
import sqlite3
import pandas as pd

In [8]:
sdm_conn = sqlite3.connect(r"C:\Gebruikers\doaaj\Semester 4\DEAL_portfolio\BikeToDrive.db")
db1_conn = sqlite3.connect("BikeToDrive_1_Accessoireverkoop.db")
db2_conn = sqlite3.connect("BikeToDrive_2_Fietsverkoop.db")
db3_conn = sqlite3.connect("BikeToDrive_3_Onderhoud.db")
db4_conn = sqlite3.connect("BikeToDrive_4_Accessoire_Inkoop.db")
db5_conn = sqlite3.connect("BikeToDrive_5_Fiets_Inkoop.db")

OperationalError: unable to open database file

In [4]:
conn = sqlite3.connect("BikeToDrive_2_Fietsverkoop.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('Fiets_Verkoop',), ('Fiets',), ('Monteur',), ('Fabrikant',), ('Filiaal',), ('Klant',)]


In [10]:
tables = [
    "Filiaal",
    "Leverancier",
    "Fabrikant",
    "Klant",
    "Monteur",
    "Accessoire",
    "Fiets",
    "Accessoire_Verkoop",
    "Fiets_Verkoop",
    "Accessoire_Inkoop",
    "Fiets_Inkoop",
    "Onderhoud"
]

tables = tables[::-1]
# Reverse the list: the last added table should be emptied first to avoid foreign key issues

sdm_cursor = sdm_conn.cursor()

for table in tables:
    delete_statement = f"DELETE FROM {table};"

    try:
        sdm_cursor.execute(delete_statement)
    except sqlite3.Error:
        print(f"FAILED: {delete_statement}")
        continue

sdm_conn.commit()

print("SDM tables cleared.")



SDM tables cleared.


In [12]:
def copy_table(source_conn, target_conn, table_name, key_column):
    source_df = pd.read_sql_query(f"SELECT * FROM {table_name};", source_conn)
    target_df = pd.read_sql_query(f"SELECT {key_column} FROM {table_name};", target_conn)

    if target_df.empty:
        new_rows = source_df
    else:
        new_rows = source_df[~source_df[key_column].isin(target_df[key_column])]

    if not new_rows.empty:
        new_rows.to_sql(table_name, target_conn, if_exists="append", index=False)
        print(f"{len(new_rows)} rows loaded into {table_name}")
    else:
        print(f"No new rows for {table_name}")

In [13]:
# Filiaal
copy_table(db1_conn, sdm_conn, "Filiaal", "filiaalnr")
copy_table(db2_conn, sdm_conn, "Filiaal", "filiaalnr")
copy_table(db3_conn, sdm_conn, "Filiaal", "filiaalnr")

# Klant
copy_table(db1_conn, sdm_conn, "Klant", "klantnr")
copy_table(db2_conn, sdm_conn, "Klant", "klantnr")

# Leverancier
copy_table(db1_conn, sdm_conn, "Leverancier", "leveranciernr")

# Fabrikant
copy_table(db2_conn, sdm_conn, "Fabrikant", "fabrikantnr")
copy_table(db3_conn, sdm_conn, "Fabrikant", "fabrikantnr")

4 rows loaded into Filiaal
No new rows for Filiaal
1 rows loaded into Filiaal
20 rows loaded into Klant
5 rows loaded into Klant
5 rows loaded into Leverancier
10 rows loaded into Fabrikant
1 rows loaded into Fabrikant


In [ ]:
pd.read_sql_query("SELECT * FROM Filiaal;", sdm_conn) #check if data is loaded

,filiaalnr,naam,adres,provincie
0,1,BikeWorld Amsterdam,Prinsengracht 100,Noord-Holland
1,2,FietsGigant Haarlem,Grote Markt 22,Noord-Holland
2,3,FietsExpress Rotterdam,Coolsingel 55,Zuid-Holland
3,4,CycleCity Leiden,Breestraat 48,Zuid-Holland
4,5,ZaanFiets Zaandam,Damstraat 5,Noord-Holland


In [15]:
# Monteur
copy_table(db1_conn, sdm_conn, "Monteur", "monteurnr")
copy_table(db2_conn, sdm_conn, "Monteur", "monteurnr")
copy_table(db3_conn, sdm_conn, "Monteur", "monteurnr")

# Accessoire
copy_table(db1_conn, sdm_conn, "Accessoire", "accessoirenr")

# Fiets
copy_table(db2_conn, sdm_conn, "Fiets", "fietsnr")
copy_table(db3_conn, sdm_conn, "Fiets", "fietsnr")

10 rows loaded into Monteur
No new rows for Monteur
5 rows loaded into Monteur
10 rows loaded into Accessoire
75 rows loaded into Fiets
No new rows for Fiets


In [ ]:
pd.read_sql_query("SELECT * FROM Monteur;", sdm_conn)
pd.read_sql_query("SELECT * FROM Accessoire;", sdm_conn)
pd.read_sql_query("SELECT * FROM Fiets;", sdm_conn) #check if data is loaded

,fietsnr,soort,merk,type,standaardprijs,inkoopprijs,kleur,fabrikant
0,1,Stadsfiets,SpeedCycle,STA-240,962.33,815.25,Groen,1
1,2,Mountainbike,SpeedCycle,MOU-394,890.64,733.06,Zwart,1
2,3,Mountainbike,SpeedCycle,MOU-341,547.79,472.47,Geel,1
3,4,Racefiets,SpeedCycle,RAC-241,991.75,787.67,Groen,1
4,5,Mountainbike,SpeedCycle,MOU-566,879.44,778.29,Geel,1
...,...,...,...,...,...,...,...,...
70,71,Elektrische fiets,FastFrame,ELE-120,731.83,576.62,Grijs,9
71,72,Racefiets,FastFrame,RAC-797,399.31,327.02,Rood,9
72,73,Stadsfiets,OranjeFiets,STA-974,731.99,622.99,Geel,10
73,74,Racefiets,OranjeFiets,RAC-701,589.50,513.51,Geel,10


In [17]:
# Accessoire_Verkoop
copy_table(db1_conn, sdm_conn, "Accessoire_Verkoop", "accessoire_verkoopnr")

# Fiets_Verkoop
copy_table(db2_conn, sdm_conn, "Fiets_Verkoop", "fiets_verkoopnr")

# Accessoire_Inkoop
copy_table(db4_conn, sdm_conn, "Accessoire_Inkoop", "inkoopnr")

# Fiets_Inkoop
copy_table(db5_conn, sdm_conn, "Fiets_Inkoop", "inkoopnr")

# Onderhoud
copy_table(db3_conn, sdm_conn, "Onderhoud", "onderhoudnr")

100 rows loaded into Accessoire_Verkoop
150 rows loaded into Fiets_Verkoop
50 rows loaded into Accessoire_Inkoop
100 rows loaded into Fiets_Inkoop
50 rows loaded into Onderhoud


In [ ]:
pd.read_sql_query("SELECT * FROM Accessoire_Verkoop;", sdm_conn)
pd.read_sql_query("SELECT * FROM Fiets_Verkoop;", sdm_conn)
pd.read_sql_query("SELECT * FROM Accessoire_Inkoop;", sdm_conn)
pd.read_sql_query("SELECT * FROM Fiets_Inkoop;", sdm_conn)
pd.read_sql_query("SELECT * FROM Onderhoud;", sdm_conn) #check if data is loaded

,onderhoudnr,datum,starttijd,eindtijd,fiets,monteur
0,1,2024-12-28,15:00:00.0000000,16:00:00.0000000,22,14
1,2,2024-02-05,14:00:00.0000000,15:00:00.0000000,18,8
2,3,2024-08-01,10:00:00.0000000,11:00:00.0000000,3,14
3,4,2024-11-16,10:00:00.0000000,11:00:00.0000000,11,2
4,5,2024-08-01,11:00:00.0000000,12:00:00.0000000,60,15
5,6,2024-12-08,13:00:00.0000000,14:00:00.0000000,18,13
6,7,2024-05-14,11:00:00.0000000,12:00:00.0000000,4,8
7,8,2024-10-10,09:00:00.0000000,10:00:00.0000000,67,3
8,9,2024-06-29,08:00:00.0000000,09:00:00.0000000,18,12
9,10,2024-08-21,08:00:00.0000000,09:00:00.0000000,33,2


In [19]:
for table in [
    "Filiaal",
    "Klant",
    "Leverancier",
    "Fabrikant",
    "Monteur",
    "Accessoire",
    "Fiets",
    "Accessoire_Verkoop",
    "Fiets_Verkoop",
    "Accessoire_Inkoop",
    "Fiets_Inkoop",
    "Onderhoud"
]:
    query = f"SELECT COUNT(*) AS aantal FROM {table};"
    df = pd.read_sql_query(query, sdm_conn)
    print(f"{table}: {df.iloc[0,0]}")

Filiaal: 5
Klant: 25
Leverancier: 5
Fabrikant: 11
Monteur: 15
Accessoire: 10
Fiets: 75
Accessoire_Verkoop: 100
Fiets_Verkoop: 150
Accessoire_Inkoop: 50
Fiets_Inkoop: 100
Onderhoud: 50


In [22]:
pd.read_sql_query("SELECT * FROM Filiaal LIMIT 5;", sdm_conn)
pd.read_sql_query("SELECT * FROM Monteur LIMIT 5;", sdm_conn)
pd.read_sql_query("SELECT * FROM Fiets LIMIT 5;", sdm_conn)
pd.read_sql_query("SELECT * FROM Onderhoud LIMIT 5;", sdm_conn) #check if data is loaded

,onderhoudnr,datum,starttijd,eindtijd,fiets,monteur
0,1,2024-12-28,15:00:00.0000000,16:00:00.0000000,22,14
1,2,2024-02-05,14:00:00.0000000,15:00:00.0000000,18,8
2,3,2024-08-01,10:00:00.0000000,11:00:00.0000000,3,14
3,4,2024-11-16,10:00:00.0000000,11:00:00.0000000,11,2
4,5,2024-08-01,11:00:00.0000000,12:00:00.0000000,60,15
